# Top 10 + Ranks 21–40 宏观意图聚类

## tl;dr

- 按论文距离与 average linkage，在主切点 `D ≤ 0.12` 得到 **677** 个临时路线族；其中 **566** 个为单例。
- 最大族有 **1,764** 个座位（51.9%）；前 10 族覆盖 **73.6%**。
- 树对原始成对距离的 cophenetic correlation 为 **0.9744**；完全相同的特征对占 **14.5%**。
- 编号使用 `N001…`，只是当前新池内部的临时编号；尚未与历史 `G001…G275` 对齐。

## Context & Methods

本次只聚类已经恢复出的 3,397 个座位，不执行参赛 Agent，也不把旧族标签预先灌入模型。距离为 `D = 0.45C + 0.35G + 0.20T`，其中 C 比较五个锚点的生产构成，G 比较格位类别，T 比较十个阶段的累计宏观动作；随后使用 average linkage。

### Key Assumptions

- 锚点为 168、288、432、576、719；动作阶段宽度为 72。
- 主切点 0.12 沿用文章口径，0.04–0.20 只用于敏感性检查。
- `N###` 按族规模降序、再按历史奖励中位数降序排序，不代表历史 `G###`。
- medoid 是族内平均距离最小的真实座位；共识阶段表是逐单元中位数四舍五入后的描述性原型。

## Data

### 1. Load clustering artifacts

In [1]:
from pathlib import Path
import csv, json, sys
from collections import Counter
feature_dir = (Path.cwd().parent / 'input').resolve()
cluster_dir = Path.cwd().resolve()
local_deps = cluster_dir.parents[1] / '.python_deps'
if local_deps.exists() and str(local_deps) not in sys.path: sys.path.insert(0, str(local_deps))
import numpy as np
def read_csv(name):
    with (cluster_dir / name).open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))
summary = json.loads((cluster_dir / 'cluster_summary_v1.json').read_text(encoding='utf-8'))
features = np.load(feature_dir / 'intent_features_v1.npz')
distance = np.load(cluster_dir / 'macro_intent_distance_v1.npy', mmap_mode='r')
linkage_matrix = np.load(cluster_dir / 'average_linkage_v1.npy')
assignments = read_csv('cluster_assignments_v1.csv')
profiles = read_csv('cluster_profiles_v1.csv')
submission_summary = read_csv('submission_cluster_summary_v1.csv')
sensitivity = read_csv('threshold_sensitivity_v1.csv')
{'seats': len(assignments), 'families': len(profiles), 'pairs': len(distance), 'submissions': len(submission_summary)}

{'seats': 3397, 'families': 677, 'pairs': 5768106, 'submissions': 30}

### 2. Validate grain, alignment, hierarchy, and cuts

In [2]:
n = len(assignments)
family_sizes = Counter(row['provisional_family_id'] for row in assignments)
profile_sizes = {row['family_id']: int(row['size']) for row in profiles}
family_counts = [int(row['family_count']) for row in sensitivity]
singleton_counts = [int(row['singleton_count']) for row in sensitivity]
checks = {
    'seat_ids_unique': len({(r['episode_id'], r['player_index'], r['submission_id']) for r in assignments}) == n,
    'seat_feature_alignment': [r['seat_index'] for r in assignments] == [str(i) for i in range(n)],
    'condensed_pair_count': len(distance) == n * (n - 1) // 2,
    'distance_finite_nonnegative': bool(np.isfinite(distance).all() and (distance >= 0).all()),
    'linkage_shape': linkage_matrix.shape == (n - 1, 4),
    'linkage_monotone': bool((np.diff(linkage_matrix[:, 2]) >= -1e-12).all()),
    'family_sizes_reconcile': family_sizes == profile_sizes,
    'one_medoid_per_family': sum(int(r['is_medoid']) for r in assignments) == len(profiles),
    'sensitivity_monotone': all(a >= b for a, b in zip(family_counts, family_counts[1:])) and all(a >= b for a, b in zip(singleton_counts, singleton_counts[1:])),
}
assert all(checks.values()), checks
checks

{'seat_ids_unique': True,
 'seat_feature_alignment': True,
 'condensed_pair_count': True,
 'distance_finite_nonnegative': True,
 'linkage_shape': True,
 'linkage_monotone': True,
 'family_sizes_reconcile': True,
 'one_medoid_per_family': True,
 'sensitivity_monotone': True}

### 3. Independently recompute sampled pair distances

In [3]:
layout = features['layout']
cumulative = features['cumulative_actions'].astype(np.int32)
def condensed_index(n, i, j):
    if i > j: i, j = j, i
    return n * i - i * (i + 1) // 2 + (j - i - 1)
def direct_distance(i, j):
    c_parts, g_parts, t_parts = [], [], []
    for anchor in range(5):
        ci = np.bincount(layout[i, anchor], minlength=11)[1:11]
        cj = np.bincount(layout[j, anchor], minlength=11)[1:11]
        c_parts.append(np.abs(ci - cj).sum() / max(8, ci.sum(), cj.sum()))
        union = (layout[i, anchor] != 0) | (layout[j, anchor] != 0)
        g_parts.append((((layout[i, anchor] != layout[j, anchor]) & union).sum() / union.sum()) if union.any() else 0.0)
    for stage in range(10):
        qi, qj = cumulative[i, stage], cumulative[j, stage]
        t_parts.append(np.abs(qi - qj).sum() / max(6, qi.sum(), qj.sum()))
    return 0.45 * np.mean(c_parts) + 0.35 * np.mean(g_parts) + 0.20 * np.mean(t_parts)
rng = np.random.default_rng(20260825)
pairs = rng.integers(0, n, size=(200, 2))
pairs = [(int(i), int(j)) for i, j in pairs if i != j]
errors = [abs(direct_distance(i, j) - float(distance[condensed_index(n, i, j)])) for i, j in pairs]
spot_check = {'pairs_checked': len(errors), 'max_abs_error': max(errors), 'mean_abs_error': float(np.mean(errors))}
assert spot_check['max_abs_error'] < 2e-7
spot_check

{'pairs_checked': 199,
 'max_abs_error': np.float64(5.1228619124543684e-08),
 'mean_abs_error': 8.35067155401399e-09}

## Results

### 4. Main cut and sensitivity

In [4]:
{
    'main_cut': summary['main_cut'],
    'distance_summary': summary['distance'],
    'family_size_summary': summary['family_size'],
    'cophenetic_correlation': float(__import__('scipy.cluster.hierarchy', fromlist=['cophenet']).cophenet(linkage_matrix, distance)[0]),
    'zero_distance_pair_share': float(np.count_nonzero(distance == 0) / len(distance)),
}

{'main_cut': {'threshold': '0.12',
              'family_count': 677,
              'singleton_count': 566,
              'largest_family': 1764,
              'top10_coverage': 0.7359434795407712,
              'top56_coverage': 0.8010008831321754},
 'distance_summary': {'min': 0.0,
                      'median': 0.31780296564102173,
                      'mean': 0.3068877394878682,
                      'max': 1.005853295326233},
 'family_size_summary': {'median': 1.0,
                         'mean': 5.017725258493353,
                         'p90': 2.0,
                         'max': 1764},
 'cophenetic_correlation': 0.9744005139608586,
 'zero_distance_pair_share': 0.14522479302564828}

### 5. Visual overview

In [5]:
from PIL import Image, ImageDraw, ImageFont
canvas = Image.new('RGB', (1400, 760), 'white')
draw = ImageDraw.Draw(canvas)
font_path = r'C:\Windows\Fonts\arial.ttf'
bold_path = r'C:\Windows\Fonts\arialbd.ttf'
font = ImageFont.truetype(font_path, 22)
small = ImageFont.truetype(font_path, 18)
title = ImageFont.truetype(bold_path, 28)
ink, grid, blue, blue_open, gold = '#263238', '#DDE3E8', '#1769AA', '#80B7D8', '#D99A2B'
draw.text((60, 30), 'Macro-intent clustering overview', font=title, fill=ink)
draw.text((60, 70), '3,397 seats; average linkage; provisional N families', font=small, fill='#60727D')
# Panel A: sensitivity line chart
x0, y0, x1, y1 = 80, 150, 720, 620
draw.text((x0, 105), 'Family counts across distance thresholds', font=font, fill=ink)
thresholds = [float(r['threshold']) for r in sensitivity]
families = [int(r['family_count']) for r in sensitivity]
singletons = [int(r['singleton_count']) for r in sensitivity]
ymax = 1000
for tick in range(0, ymax + 1, 200):
    y = y1 - (y1-y0) * tick / ymax
    draw.line((x0, y, x1, y), fill=grid, width=1)
    draw.text((x0-55, y-10), str(tick), font=small, fill='#60727D')
xs = [x0 + (x1-x0) * i / (len(thresholds)-1) for i in range(len(thresholds))]
for x, value in zip(xs, thresholds):
    draw.text((x-18, y1+15), f'{value:.2f}', font=small, fill='#60727D')
def ycoord(value): return y1 - (y1-y0) * value / ymax
for values, color, width in [(families, blue, 4), (singletons, blue_open, 4)]:
    points = [(x, ycoord(v)) for x, v in zip(xs, values)]
    draw.line(points, fill=color, width=width)
    for x, y in points: draw.ellipse((x-5, y-5, x+5, y+5), fill='white', outline=color, width=3)
draw.line((x0, y0, x0, y1), fill=ink, width=2); draw.line((x0, y1, x1, y1), fill=ink, width=2)
draw.line((x0+20, 660, x0+70, 660), fill=blue, width=4); draw.text((x0+80, 648), 'Families', font=small, fill=ink)
draw.line((x0+220, 660, x0+270, 660), fill=blue_open, width=4); draw.text((x0+280, 648), 'Singletons', font=small, fill=ink)
# Panel B: part-to-whole concentration at 0.12
bx0, bx1 = 810, 1330
draw.text((bx0, 105), 'Seat share at the main cut (D <= 0.12)', font=font, fill=ink)
sizes = sorted((int(r['size']) for r in profiles), reverse=True)
parts = [('N001', sizes[0], blue), ('N002-N010', sum(sizes[1:10]), gold), ('Long tail', sum(sizes[10:]), '#B0BEC5')]
bar_y, bar_h, current = 230, 90, bx0
for label, value, color in parts:
    width = (bx1-bx0) * value / n
    draw.rectangle((current, bar_y, current+width, bar_y+bar_h), fill=color, outline='white', width=2)
    current += width
legend_y = 390
for label, value, color in parts:
    draw.rectangle((bx0, legend_y, bx0+24, legend_y+24), fill=color)
    draw.text((bx0+40, legend_y-3), f'{label}: {value:,} ({value/n:.1%})', font=font, fill=ink)
    legend_y += 58
draw.text((bx0, 580), 'The long tail contains 667 families.', font=small, fill='#60727D')
draw.text((bx0, 615), 'Source: reconstructed replay macro intents.', font=small, fill='#60727D')
chart_path = cluster_dir / 'cluster_overview_v1.png'
canvas.save(chart_path)
{'chart': str(chart_path), 'size_pixels': canvas.size}

{'chart': 'C:\\Users\\DHU_Z\\OneDrive\\文档\\ChatGPT\\kaggleculture\\artifacts\\top10_plus_ranks21_40_clusters_v1\\cluster_overview_v1.png',
 'size_pixels': (1400, 760)}

![Cluster overview](cluster_overview_v1.png)

### 6. Largest families and submission-level tail

In [6]:
largest_families = [{k: row[k] for k in ['family_id','size','share','median_reward','win_rate','unique_submissions','medoid_team','medoid_seat_id']} for row in profiles[:10]]
long_tail_by_submission = sorted(
    [{k: row[k] for k in ['rank','team','seat_count','family_count','singleton_seats','singleton_share','dominant_family_id','dominant_family_share']} for row in submission_summary],
    key=lambda row: int(row['singleton_seats']), reverse=True
)
{'largest_families': largest_families, 'submissions_by_singletons': long_tail_by_submission}

{'largest_families': [{'family_id': 'N001',
                       'size': '1764',
                       'share': '0.5192817191639683',
                       'median_reward': '82131.0',
                       'win_rate': '0.6757369614512472',
                       'unique_submissions': '25',
                       'medoid_team': 'Arman Tuganbaev',
                       'medoid_seat_id': '94478362:1:55617399'},
                      {'family_id': 'N002',
                       'size': '260',
                       'share': '0.07653812187224021',
                       'median_reward': '101460.5',
                       'win_rate': '0.7461538461538462',
                       'unique_submissions': '24',
                       'medoid_team': 'Arman Tuganbaev',
                       'medoid_seat_id': '94464220:1:55617399'},
                      {'family_id': 'N003',
                       'size': '247',
                       'share': '0.0727112157786282',
                       'med

## Takeaways

- 当前池并不是均匀分成数百族，而是一个跨 25 个 submission 的主流宏观骨架，加上少量中型族和很长的个性化尾部。
- 566 个单例全部来自 6 个 submission，主要集中在 rank 3、rank 1、rank 2 与 rank 39；因此高族数主要反映这些 Agent 的逐局变化，不是所有队伍都同样碎片化。
- 把切点从 0.12 放宽到 0.20，族数仍为 377，说明长尾对切点具有持续性；下一步不应只靠放宽阈值压缩族数。
- 这批 `N###` 适合作为新样本内部的结构地图。正式恢复历史宏观意图时，应把它们与旧 275 族联合重聚类或做 medoid 对齐后再授予 `G###`。